# Tutorial: Plotting and Reporting Packages

Detailed step-by-step workflow notebook for this SD-dMFA repository.


## Audience, Prerequisites, Outcomes

**Audience**
- Analysts producing standardized figures for reporting.

**Prerequisites**
- Python 3.11+ environment for this repo.
- `pip install -e ".[dev]"` completed.
- Notebook executed from repository root or a subfolder.

**Outcomes**
- Generate subset and indicator panel packages.
- Include optional end-use detail panels.
- Run observed-vs-modeled visual diagnostics.


## Outline

1. Run subset panel generator
2. Run end-use detail source + panel inclusion
3. Inspect generated plot directories
4. Run observed-vs-modeled plot utility
5. Collect plot artifacts for reporting


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
from pathlib import Path
from textwrap import dedent

import pandas as pd
import matplotlib.pyplot as plt

try:
    from crm_model.common.io import load_run_config
except Exception:
    load_run_config = None

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)


In [ ]:
DRY_RUN = False
RUN_HEAVY = False
RUN_PLOTS = False
RUN_CALIBRATION = False
RUN_AUDIT = False

CONFIG = "configs/runs/mvp.yml"
EXAMPLE_VARIANT = "baseline"


In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "configs").exists() and (cand / "src").exists():
            return cand
    raise RuntimeError("Could not locate repo root from current working directory.")


def sh(cmd: str, *, cwd: Path, check: bool = True) -> subprocess.CompletedProcess | None:
    print(f"$ {cmd}")
    if DRY_RUN:
        return None
    cp = subprocess.run(cmd, cwd=str(cwd), shell=True, text=True, capture_output=True)
    if cp.stdout.strip():
        print(cp.stdout)
    if cp.stderr.strip():
        print(cp.stderr)
    if check and cp.returncode != 0:
        raise RuntimeError(f"Command failed ({cp.returncode}): {cmd}")
    return cp


def latest_dir(base: Path) -> Path | None:
    if not base.exists():
        return None
    cands = [p for p in base.iterdir() if p.is_dir() and p.name != "_archive"]
    return sorted(cands)[-1] if cands else None


def load_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        print(f"Missing: {path}")
        return pd.DataFrame()
    return pd.read_csv(path)


REPO = find_repo_root()
CONFIG_PATH = (REPO / CONFIG).resolve()
CONFIG_STEM = CONFIG_PATH.stem
print("Repo:", REPO)
print("Config:", CONFIG_PATH)


## Step 1: Generate subset/indicator panel package


In [ ]:
if RUN_PLOTS:
    _ = sh(f"python scripts/analysis/plots/plot_scenario_subset_panels.py --config {CONFIG}", cwd=REPO)
else:
    print("Set RUN_PLOTS=True to generate plot packages")


## Step 2: Optional end-use source and detail panels


In [ ]:
if RUN_PLOTS:
    _ = sh(f"python scripts/analysis/plots/plot_stock_in_use_by_end_use_region_scenarios.py --config {CONFIG}", cwd=REPO)
    sources = sorted((REPO / "outputs" / "analysis" / "stock_in_use_by_end_use_region_scenarios").glob("**/stock_in_use_by_end_use_region_scenario.csv"))
    if sources:
        src = sources[-1]
        print("Using end-use source:", src)
        _ = sh(f"python scripts/analysis/plots/plot_scenario_subset_panels.py --config {CONFIG} --end-use-source {src}", cwd=REPO)


## Step 3: Inspect generated plot files


In [ ]:
plot_root = REPO / "outputs" / "analysis" / "scenario_comparison" / CONFIG_STEM / "latest" / "plots"
print("Plot root:", plot_root)
if plot_root.exists():
    for sub in ["subset_panels", "indicator_panels", "end_use_detail"]:
        p = plot_root / sub
        n = len(list(p.glob("*.png"))) if p.exists() else 0
        print(sub, "png files:", n)


## Step 4: Show one generated image inline (if present)


In [ ]:
import matplotlib.image as mpimg

subset_dir = REPO / "outputs" / "analysis" / "scenario_comparison" / CONFIG_STEM / "latest" / "plots" / "subset_panels"
imgs = sorted(subset_dir.glob("*.png"))
if imgs:
    img = mpimg.imread(imgs[0])
    plt.figure(figsize=(12, 6))
    plt.imshow(img)
    plt.axis("off")
    plt.title(imgs[0].name)
    plt.show()
else:
    print("No subset panel image found yet")


## Step 5: Observed vs modeled plotting utility


In [ ]:
cmd = (
    f"python scripts/analysis/plots/plot_observed_vs_model.py "
    f"--config {CONFIG} --phase full --baseline-variant baseline --calibrated-variant calibrated"
)
print(cmd)
if RUN_PLOTS:
    _ = sh(cmd, cwd=REPO, check=False)


## Pitfalls

- Plot scripts rely on existing run outputs; generate runs first.
- Use matching config stem for compare/plot roots to avoid stale visual mixes.


## Exercises

1. Repeat this workflow with `CONFIG=configs/runs/r-strategies.yml`.
2. Record one thing that changed and why.
3. Add one guardrail/check specific to your team workflow.


In [ ]:
# Exercise answer scaffold
pass
